# 💳 Predicting Loan Default with Machine Learning
### A Fintech Tutorial Using Real LendingClub Data

---

**Welcome back!** In the last tutorial, we predicted March Madness results. Now we're applying the exact same tools to a problem that fintech companies deal with every day: **will this borrower pay back their loan?**

We're using a real dataset from **LendingClub** — one of the largest peer-to-peer lending platforms in the US. Every row is a real loan made to a real person. Every feature is something LendingClub actually knew at the time they decided whether to approve the loan.

### The Business Problem

Imagine you're a fintech lender. You have to decide: *should I approve this loan application?*

- If you approve too many risky loans → borrowers default → you lose money
- If you reject too many good loans → you miss revenue → competitors beat you

A well-calibrated machine learning model helps you make this tradeoff smarter and faster than any human could at scale.

---

### 🏀 → 💳 The Connection to the Last Tutorial

| Basketball | Credit Default |
|---|---|
| Did they make the Sweet 16? | Did they default on the loan? |
| Team stats (offense, defense, wins...) | Borrower profile (income, FICO, DTI...) |
| Logistic Regression + Decision Tree | Same exact models! |
| Precision / Recall tradeoff | Even MORE critical here — false negatives cost real money |

---

## 📦 Step 1: Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_score,
    recall_score, f1_score
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Libraries loaded!")

✅ Libraries loaded!


---
## 📊 Step 2: Load and First Look at the Data

This dataset comes from **LendingClub**, a real peer-to-peer lending platform. It contains ~26,500 loans with 151 columns of information about each borrower and loan.

In [2]:
# Load the data — update the filename/path if needed
df = pd.read_excel('LC_Sample.xlsx')

print(f"Dataset shape: {df.shape[0]:,} loans × {df.shape[1]} columns")
print(f"\nFirst look at key columns:")
df[['loan_amnt', 'term', 'int_rate', 'grade', 'annual_inc',
    'purpose', 'home_ownership', 'loan_status']].head(10)

Dataset shape: 26,543 loans × 151 columns

First look at key columns:


,loan_amnt,term,int_rate,grade,annual_inc,purpose,home_ownership,loan_status
0,8400,36 months,0.0975,B,66000.0,debt_consolidation,MORTGAGE,Fully Paid
1,12000,36 months,0.0789,A,45000.0,credit_card,OWN,Current
2,28000,36 months,0.0739,A,60000.0,debt_consolidation,RENT,Current
3,10000,36 months,0.1367,C,70000.0,other,RENT,Current
4,20000,36 months,0.1199,C,49000.0,debt_consolidation,MORTGAGE,Fully Paid
5,13625,60 months,0.1531,C,50000.0,credit_card,MORTGAGE,Fully Paid
6,9000,36 months,0.0916,B,60000.0,debt_consolidation,RENT,Current
7,15000,60 months,0.1953,D,63890.0,debt_consolidation,RENT,Charged Off
8,18000,36 months,0.1299,C,53000.0,debt_consolidation,RENT,Current
9,12000,60 months,0.1367,C,68000.0,credit_card,MORTGAGE,Late (16-30 days)


In [ ]:
# What does loan_status look like? This is our raw target variable.
print("Loan Status Distribution:")
status_counts = df['loan_status'].value_counts()
print(status_counts)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#4c78a8', '#54a24b', '#e45756', '#f58518', '#eeca3b', '#72b7b2', '#b279a2']
bars = ax.barh(status_counts.index, status_counts.values, color=colors, edgecolor='white')

for bar, count in zip(bars, status_counts.values):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f'{count:,}', va='center', fontsize=10, fontweight='bold')

ax.set_title('What Happened to Each Loan?', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Loans')
plt.tight_layout()
plt.show()

---
## 🎯 Step 3: Define "Default" — Creating Our Target Variable

The `loan_status` column has 7 categories, but our model needs a simple yes/no answer: **did this loan default?**

We need to make a judgment call here. This is one of the most important decisions in any ML project — how we define the outcome.

**Our definition of default:**
- **1 (Defaulted)** = `Charged Off`, `Default`, `Late (31-120 days)`, `Late (16-30 days)`
- **0 (Good)** = `Fully Paid`

> ⚠️ **What about "Current" loans?** We'll drop them. Why? Because we don't know the final outcome yet — they're still being repaid. Including them would be like judging a basketball game at halftime. For training a model, we only want **completed outcomes**.

> ⚠️ **"In Grace Period"** — also dropped, since the outcome is still uncertain.

In [ ]:
# Keep only loans with known final outcomes
completed_statuses = ['Fully Paid', 'Charged Off', 'Default',
                      'Late (31-120 days)', 'Late (16-30 days)']

df_model = df[df['loan_status'].isin(completed_statuses)].copy()
print(f"Loans with known outcomes: {len(df_model):,}")
print(f"  (Removed {len(df) - len(df_model):,} 'Current' and 'In Grace Period' loans)")

# Create binary target: 1 = defaulted, 0 = fully paid
default_statuses = ['Charged Off', 'Default', 'Late (31-120 days)', 'Late (16-30 days)']
df_model['defaulted'] = df_model['loan_status'].apply(
    lambda x: 1 if x in default_statuses else 0
)

n_default = df_model['defaulted'].sum()
n_total = len(df_model)
print(f"\nDefault rate: {n_default:,} / {n_total:,} = {n_default/n_total:.1%}")
print("\n💡 This means roughly 1 in 3 loans in our completed set defaulted.")
print("   That's actually high — real default rates are typically 5-15%.")
print("   This is because late-stage loans are overrepresented in the data.")

In [ ]:
# Visualize the target split
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
counts = df_model['defaulted'].value_counts().sort_index()
axes[0].pie(
    counts.values,
    labels=['Fully Paid', 'Defaulted'],
    colors=['#54a24b', '#e45756'],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title('Loan Outcomes (Completed Loans)', fontweight='bold')

# Default rate by grade
grade_default = df_model.groupby('grade')['defaulted'].mean().reset_index()
grade_default.columns = ['grade', 'default_rate']
grade_order = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
grade_default = grade_default.set_index('grade').reindex(grade_order).reset_index()

bars = axes[1].bar(grade_default['grade'], grade_default['default_rate'],
                   color=sns.color_palette('RdYlGn_r', 7), edgecolor='white')
for bar, rate in zip(bars, grade_default['default_rate']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{rate:.0%}', ha='center', fontsize=10, fontweight='bold')

axes[1].set_title('Default Rate by Loan Grade', fontweight='bold')
axes[1].set_xlabel('LendingClub Grade (A = safest, G = riskiest)')
axes[1].set_ylabel('Default Rate')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))

plt.suptitle('Understanding Default in Our Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Notice how default rates climb steadily from Grade A to Grade G.")
print("   LendingClub's own grading system is already doing a good job —")
print("   our model needs to do even better using the underlying raw data.")

---
## 🔍 Step 4: Exploratory Data Analysis

Let's explore the most important features and understand what separates defaulted loans from good ones. This builds our intuition before we hand things off to the model.

### Key Features We'll Use

| Feature | Description | Why it matters |
|---------|-------------|----------------|
| `loan_amnt` | Loan amount requested | Larger loans = more repayment burden |
| `int_rate` | Interest rate on the loan | Higher rate = riskier borrower (LendingClub prices risk in) |
| `annual_inc` | Borrower's annual income | Higher income = more ability to repay |
| `dti` | Debt-to-Income ratio | How much existing debt vs income (critical risk signal) |
| `fico_range_low` | Credit score (low end of range) | Higher FICO = better credit history |
| `open_acc` | Number of open credit lines | Too many can indicate over-leverage |
| `revol_util` | Revolving credit utilization (%) | How much of available credit is being used |
| `delinq_2yrs` | Delinquencies in past 2 years | Past behavior predicts future behavior |
| `inq_last_6mths` | Credit inquiries in last 6 months | Many inquiries = shopping for credit = red flag |
| `pub_rec` | Public derogatory records | Bankruptcies, tax liens, etc. |
| `term` | 36 or 60 month loan | Longer term = more exposure |
| `home_ownership` | Rent / Own / Mortgage | Proxy for financial stability |

In [ ]:
# Compare average stats: defaulted vs fully paid
key_stats = ['loan_amnt', 'int_rate', 'annual_inc', 'dti',
             'fico_range_low', 'revol_util', 'delinq_2yrs', 'inq_last_6mths']

comparison = df_model.groupby('defaulted')[key_stats].median().round(3)
comparison.index = ['Fully Paid', 'Defaulted']
comparison['int_rate'] = comparison['int_rate'].apply(lambda x: f"{x:.1%}")
comparison['revol_util'] = comparison['revol_util'].apply(lambda x: f"{x:.1%}")
comparison['annual_inc'] = comparison['annual_inc'].apply(lambda x: f"${x:,.0f}")
comparison['loan_amnt'] = comparison['loan_amnt'].apply(lambda x: f"${x:,.0f}")

print("Median stats by outcome:")
comparison

In [ ]:
# Four key charts side by side
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

plot_configs = [
    ('int_rate', 'Interest Rate', True, 'Interest Rate on Loan'),
    ('fico_range_low', 'FICO Score', False, 'Borrower Credit Score (FICO)'),
    ('dti', 'Debt-to-Income Ratio', True, 'Debt-to-Income Ratio (DTI)'),
    ('annual_inc', 'Annual Income', False, 'Annual Income'),
]

for ax, (col, xlabel, cap_needed, title) in zip(axes.flatten(), plot_configs):
    plot_data = df_model[df_model[col].notna()].copy()
    
    # Cap outliers for cleaner charts
    if cap_needed:
        cap = plot_data[col].quantile(0.99)
        plot_data = plot_data[plot_data[col] <= cap]
    else:
        cap = plot_data[col].quantile(0.97)
        plot_data = plot_data[plot_data[col] <= cap]
    
    for outcome, color, label in zip([0, 1], ['#54a24b', '#e45756'],
                                      ['Fully Paid', 'Defaulted']):
        subset = plot_data[plot_data['defaulted'] == outcome][col]
        ax.hist(subset, bins=30, alpha=0.6, color=color, label=label, density=True, edgecolor='none')
    
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('How Do Defaulters Look Different from Good Borrowers?',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\n💡 Key observations:")
print("  • Defaulters tend to have HIGHER interest rates (riskier borrowers pay more)")
print("  • Defaulters tend to have LOWER FICO scores")
print("  • Defaulters tend to have HIGHER DTI ratios")
print("  • Income distributions overlap a lot — income alone isn't enough to predict default")

In [ ]:
# Default rate by loan purpose
purpose_default = df_model.groupby('purpose').agg(
    default_rate=('defaulted', 'mean'),
    count=('defaulted', 'count')
).query('count >= 50').sort_values('default_rate', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#54a24b' if r < 0.25 else '#f58518' if r < 0.35 else '#e45756'
          for r in purpose_default['default_rate']]
bars = ax.barh(purpose_default.index, purpose_default['default_rate'],
               color=colors, edgecolor='white')

for bar, (rate, count) in zip(bars, purpose_default[['default_rate', 'count']].values):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{rate:.0%}  (n={count:,})', va='center', fontsize=9)

ax.set_title('Default Rate by Loan Purpose', fontsize=13, fontweight='bold')
ax.set_xlabel('Default Rate')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.axvline(x=df_model['defaulted'].mean(), color='gray', linestyle='--', alpha=0.7,
           label=f'Overall average ({df_model["defaulted"].mean():.0%})')
ax.legend()
plt.tight_layout()
plt.show()

print("\n💡 Small business loans default at much higher rates than car loans or vacations.")
print("   Loan purpose is an important signal — the model will pick up on this.")

In [ ]:
# Default rate by home ownership and term
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Home ownership
home_default = df_model.groupby('home_ownership')['defaulted'].mean().sort_values(ascending=False)
axes[0].bar(home_default.index, home_default.values,
            color=['#e45756', '#f58518', '#54a24b'], edgecolor='white')
for i, (label, rate) in enumerate(home_default.items()):
    axes[0].text(i, rate + 0.005, f'{rate:.1%}', ha='center', fontweight='bold')
axes[0].set_title('Default Rate by Home Ownership', fontweight='bold')
axes[0].set_ylabel('Default Rate')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))

# Term
term_default = df_model.groupby('term')['defaulted'].mean()
axes[1].bar(term_default.index, term_default.values,
            color=['#4c78a8', '#e45756'], edgecolor='white', width=0.4)
for i, (label, rate) in enumerate(term_default.items()):
    axes[1].text(i, rate + 0.005, f'{rate:.1%}', ha='center', fontweight='bold')
axes[1].set_title('Default Rate by Loan Term', fontweight='bold')
axes[1].set_ylabel('Default Rate')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))

plt.tight_layout()
plt.show()

---
## 🛠️ Step 5: Prepare the Data for Modeling

Real-world data is messy. Before we can train a model, we need to:

1. **Select features** — pick the columns that were knowable at loan origination
2. **Handle missing values** — decide what to do with gaps
3. **Encode categorical variables** — convert text (like "RENT") into numbers
4. **Split into train/test sets**
5. **Scale the features**

### ⚠️ A Critical Rule: No Data Leakage!

This is one of the most common mistakes beginners make. **Data leakage** means accidentally giving the model information it wouldn't have had at the time of the decision.

For example:
- `total_pymnt` (total payments received) — this was measured AFTER the loan concluded. A model using this would be cheating!
- `recoveries` — only non-zero on defaulted loans. The model would "cheat" by just looking at this.
- `last_pymnt_amnt` — also measured after the fact

We must only use features that would be **available at the time of loan approval**.

In [ ]:
# Select features available at loan origination
# (No post-origination payment data — that would be leakage!)

numeric_features = [
    'loan_amnt',        # Loan amount
    'int_rate',         # Interest rate (LendingClub's risk pricing)
    'installment',      # Monthly payment amount
    'annual_inc',       # Borrower's annual income
    'dti',              # Debt-to-income ratio
    'fico_range_low',   # Credit score (low end)
    'open_acc',         # Number of open credit lines
    'revol_util',       # Revolving credit utilization
    'delinq_2yrs',      # Delinquencies in past 2 years
    'inq_last_6mths',   # Credit inquiries in last 6 months
    'pub_rec',          # Public derogatory records
    'revol_bal',        # Total revolving balance
    'total_acc',        # Total credit accounts
]

categorical_features = [
    'term',             # 36 or 60 months
    'home_ownership',   # RENT / OWN / MORTGAGE
    'purpose',          # Loan purpose (debt consolidation, etc.)
    'verification_status',  # Whether income was verified
]

target = 'defaulted'

all_features = numeric_features + categorical_features
model_df = df_model[all_features + [target]].copy()

print(f"Starting rows: {len(model_df):,}")
print("\nMissing values per feature:")
missing = model_df[all_features].isna().sum()
print(missing[missing > 0])

In [ ]:
# Handle missing values

# DTI: fill with median (a reasonable default for income ratios)
model_df['dti'] = model_df['dti'].fillna(model_df['dti'].median())

# revol_util: fill with median
model_df['revol_util'] = model_df['revol_util'].fillna(model_df['revol_util'].median())

# Cap extreme outliers in annual_inc and dti (likely data errors)
model_df['annual_inc'] = model_df['annual_inc'].clip(upper=model_df['annual_inc'].quantile(0.99))
model_df['dti'] = model_df['dti'].clip(upper=100)

# Drop any remaining rows with nulls (very few)
model_df = model_df.dropna()

print(f"Rows after cleaning: {len(model_df):,}")
print(f"Default rate in clean dataset: {model_df[target].mean():.1%}")
print("\n✅ Data is clean and ready for modeling!")

In [ ]:
# Encode categorical variables using One-Hot Encoding
#
# What is one-hot encoding?
# A model can't process text like "RENT" — it needs numbers.
# One-hot encoding creates a binary column for each category:
#
#   home_ownership = "RENT"     →   home_RENT=1, home_OWN=0, home_MORTGAGE=0
#   home_ownership = "OWN"      →   home_RENT=0, home_OWN=1, home_MORTGAGE=0
#   home_ownership = "MORTGAGE" →   home_RENT=0, home_OWN=0, home_MORTGAGE=1

model_df_encoded = pd.get_dummies(
    model_df,
    columns=categorical_features,
    drop_first=True  # avoids redundant columns
)

# Separate features from target
feature_cols = [c for c in model_df_encoded.columns if c != target]
X = model_df_encoded[feature_cols]
y = model_df_encoded[target]

print(f"Feature matrix shape: {X.shape}")
print(f"\nFeatures used ({len(feature_cols)} total):")
print(feature_cols)

In [ ]:
# Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y    # ensures same default rate in both splits
)

print(f"Training set: {len(X_train):,} loans")
print(f"Test set:     {len(X_test):,} loans")
print(f"\nDefault rate — Train: {y_train.mean():.1%} | Test: {y_test.mean():.1%}")
print("✅ Rates match — our split is properly balanced!")

# Scale features (fit on train only!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

---
## 🤖 Step 6: Model 1 — Logistic Regression

Just like in the basketball tutorial, we start with logistic regression. This is actually the **industry standard for credit risk modeling** — banks have been using it for decades because:

1. It's simple and fast
2. It produces a **probability** ("this loan has a 23% chance of default") not just yes/no
3. It's **explainable** — regulators require banks to explain why a loan was denied
4. It works well on linearly separable problems

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42, C=0.5)
lr_model.fit(X_train_scaled, y_train)

lr_preds = lr_model.predict(X_test_scaled)
lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]

lr_accuracy = accuracy_score(y_test, lr_preds)
lr_auc      = roc_auc_score(y_test, lr_probs)

print("✅ Logistic Regression trained!")
print(f"   Accuracy:  {lr_accuracy:.1%}")
print(f"   AUC Score: {lr_auc:.3f}")
print()
print("💡 What is AUC?")
print("   AUC (Area Under the Curve) measures how well the model separates the two classes.")
print("   0.5 = no better than random | 1.0 = perfect | 0.7+ = generally useful")

In [ ]:
# What did Logistic Regression learn? Feature coefficients
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient')

# Show only top/bottom 10 for readability
top_n = 12
coef_plot = pd.concat([coef_df.head(top_n//2), coef_df.tail(top_n//2)])

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#e45756' if c > 0 else '#54a24b' for c in coef_plot['Coefficient']]
ax.barh(coef_plot['Feature'], coef_plot['Coefficient'], color=colors, edgecolor='white')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression: What Predicts Default?\n(Top risk-increasing and risk-decreasing features)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Coefficient (Red = increases default risk | Green = decreases default risk)')
plt.tight_layout()
plt.show()

print("\n💡 Does this match your intuition?")
print("   Higher int_rate → more default risk (lenders charge more for risky borrowers)")
print("   Higher FICO score → less default risk (good credit history = better outcomes)")
print("   Higher DTI → more default risk (carrying too much debt relative to income)")

In [ ]:
# Confusion Matrix
cm_lr = confusion_matrix(y_test, lr_preds)
disp = ConfusionMatrixDisplay(cm_lr, display_labels=['Fully Paid', 'Defaulted'])

fig, ax = plt.subplots(figsize=(7, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Logistic Regression — Confusion Matrix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(classification_report(y_test, lr_preds, target_names=['Fully Paid', 'Defaulted']))

print("\n💰 Business translation:")
tn, fp, fn, tp = cm_lr.ravel()
print(f"   True Negatives  ({tn:,}): Correctly predicted good loans → APPROVED, earned interest ✅")
print(f"   True Positives  ({tp:,}): Correctly caught defaulters → DENIED, avoided loss ✅")
print(f"   False Positives ({fp:,}): Good borrowers wrongly flagged → DENIED, missed revenue ⚠️")
print(f"   False Negatives ({fn:,}): Missed defaulters → APPROVED, money lost 🚨")

---
## 🌳 Step 7: Model 2 — Decision Tree

In [ ]:
# Train Decision Tree
dt_model = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=50,
    random_state=42
)
dt_model.fit(X_train, y_train)  # trees don't need scaled data

dt_preds = dt_model.predict(X_test)
dt_probs = dt_model.predict_proba(X_test)[:, 1]

dt_accuracy = accuracy_score(y_test, dt_preds)
dt_auc      = roc_auc_score(y_test, dt_probs)

print(f"✅ Decision Tree trained!")
print(f"   Accuracy:  {dt_accuracy:.1%}")
print(f"   AUC Score: {dt_auc:.3f}")

In [ ]:
# Visualize the Decision Tree — the most intuitive model output
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(
    dt_model,
    feature_names=feature_cols,
    class_names=['Fully Paid', 'Defaulted'],
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax,
    impurity=False
)
ax.set_title('Decision Tree: The Model\'s Decision Rules\n(Read top to bottom — each node is a yes/no question)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 This is why decision trees are loved in regulated industries like banking.")
print("   You can hand this to a regulator or a customer and explain EXACTLY why")
print("   a loan was approved or denied. No black box!")

In [ ]:
# Decision Tree Feature Importance
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=True).tail(12)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(importance_df['Feature'], importance_df['Importance'],
        color='#4c78a8', edgecolor='white')
ax.set_title('Decision Tree: Top 12 Most Important Features', fontsize=12, fontweight='bold')
ax.set_xlabel('Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Decision Tree Confusion Matrix
cm_dt = confusion_matrix(y_test, dt_preds)
disp_dt = ConfusionMatrixDisplay(cm_dt, display_labels=['Fully Paid', 'Defaulted'])

fig, ax = plt.subplots(figsize=(7, 5))
disp_dt.plot(ax=ax, colorbar=False, cmap='Greens')
ax.set_title('Decision Tree — Confusion Matrix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(classification_report(y_test, dt_preds, target_names=['Fully Paid', 'Defaulted']))

---
## 📈 Step 8: The ROC Curve — A More Complete Picture of Model Performance

So far we've been evaluating models at a single threshold (50%). But in practice, you can **tune the threshold** to prioritize different goals:

- A conservative lender might set the threshold at 30% → catch more defaulters, but reject more good borrowers too
- An aggressive lender might set it at 70% → approve more loans, but take on more risk

The **ROC Curve** shows model performance across ALL possible thresholds. The **AUC (Area Under the Curve)** summarizes it in one number:
- 0.5 = random guessing (useless)
- 1.0 = perfect model
- 0.7+ = generally considered useful in practice

In [ ]:
# ROC Curve comparison
fig, ax = plt.subplots(figsize=(8, 7))

for model_name, probs, color in [
    ('Logistic Regression', lr_probs, '#4c78a8'),
    ('Decision Tree',       dt_probs, '#f28e2b')
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, color=color, linewidth=2.5,
            label=f'{model_name} (AUC = {auc:.3f})')

# Random baseline
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random Guess (AUC = 0.500)')

ax.fill_between(fpr, tpr, alpha=0)
ax.set_xlabel('False Positive Rate\n(Fraction of good loans wrongly flagged as risky)', fontsize=11)
ax.set_ylabel('True Positive Rate\n(Fraction of defaulters caught)', fontsize=11)
ax.set_title('ROC Curve: Model Performance Across All Thresholds',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

# Add quadrant annotation
ax.annotate('Ideal: High TPR,\nLow FPR',
            xy=(0.05, 0.92), fontsize=9, color='#2e7d32',
            bbox=dict(boxstyle='round', facecolor='#e8f5e9', alpha=0.8))

plt.tight_layout()
plt.show()

print("\n💡 The higher the curve (closer to top-left), the better the model.")
print("   Both models beat random guessing by a meaningful margin.")
print("   AUC of 0.70+ is typical for credit risk models on public data.")
print("   Real bank models use 100s of proprietary features and can reach 0.80+.")

---
## ⚖️ Step 9: Model Comparison

In [ ]:
# Full comparison table
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree'],
    'Accuracy':  [accuracy_score(y_test, lr_preds),  accuracy_score(y_test, dt_preds)],
    'Precision': [precision_score(y_test, lr_preds), precision_score(y_test, dt_preds)],
    'Recall':    [recall_score(y_test, lr_preds),    recall_score(y_test, dt_preds)],
    'F1-Score':  [f1_score(y_test, lr_preds),        f1_score(y_test, dt_preds)],
    'AUC':       [roc_auc_score(y_test, lr_probs),   roc_auc_score(y_test, dt_probs)],
})

results_display = results.set_index('Model')
print("Model Comparison (all metrics on the test set):")
results_display.applymap(lambda x: f"{x:.1%}" if x < 1.5 else f"{x:.3f}")

In [ ]:
# Visual comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC']
lr_scores = [results.iloc[0][m] for m in metrics]
dt_scores = [results.iloc[1][m] for m in metrics]

x = np.arange(len(metrics))
width = 0.35
fig, ax = plt.subplots(figsize=(11, 6))
bars1 = ax.bar(x - width/2, lr_scores, width, label='Logistic Regression',
               color='#4c78a8', edgecolor='white')
bars2 = ax.bar(x + width/2, dt_scores, width, label='Decision Tree',
               color='#f28e2b', edgecolor='white')

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax.set_ylim(0, 1.1)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_title('Model Comparison: Logistic Regression vs Decision Tree',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4, label='Random baseline')
plt.tight_layout()
plt.show()

---
## 🎮 Step 10: Score a New Borrower

The real payoff — let's build a function that takes a hypothetical borrower's profile and outputs a default probability from both models.

This is what happens behind the scenes every time someone applies for a loan online.

In [ ]:
def score_borrower(loan_amnt, int_rate, annual_inc, dti, fico,
                   term='36 months', home_ownership='RENT',
                   purpose='debt_consolidation', verification_status='Not Verified',
                   installment=None, open_acc=8, revol_util=0.5,
                   delinq_2yrs=0, inq_last_6mths=1, pub_rec=0,
                   revol_bal=10000, total_acc=15):
    """
    Score a hypothetical borrower and return default probabilities.
    """
    if installment is None:
        months = 36 if '36' in term else 60
        installment = (loan_amnt * int_rate / 12) / (1 - (1 + int_rate/12)**(-months))
    
    # Build a single-row dataframe matching our feature set
    raw = pd.DataFrame([{
        'loan_amnt': loan_amnt,
        'int_rate': int_rate,
        'installment': installment,
        'annual_inc': min(annual_inc, model_df['annual_inc'].quantile(0.99)),
        'dti': min(dti, 100),
        'fico_range_low': fico,
        'open_acc': open_acc,
        'revol_util': revol_util,
        'delinq_2yrs': delinq_2yrs,
        'inq_last_6mths': inq_last_6mths,
        'pub_rec': pub_rec,
        'revol_bal': revol_bal,
        'total_acc': total_acc,
        'term': term,
        'home_ownership': home_ownership,
        'purpose': purpose,
        'verification_status': verification_status,
    }])
    
    # One-hot encode to match training format
    raw_encoded = pd.get_dummies(raw, columns=categorical_features, drop_first=True)
    
    # Align columns to match training set (add missing dummy cols as 0)
    for col in feature_cols:
        if col not in raw_encoded.columns:
            raw_encoded[col] = 0
    raw_encoded = raw_encoded[feature_cols]
    
    # Score
    raw_scaled = scaler.transform(raw_encoded)
    lr_prob = lr_model.predict_proba(raw_scaled)[0, 1]
    dt_prob = dt_model.predict_proba(raw_encoded)[0, 1]
    
    # Display
    print("=" * 55)
    print("  BORROWER PROFILE")
    print("=" * 55)
    print(f"  Loan Amount:          ${loan_amnt:>10,.0f}")
    print(f"  Interest Rate:        {int_rate:>10.1%}")
    print(f"  Annual Income:        ${annual_inc:>10,.0f}")
    print(f"  Debt-to-Income (DTI): {dti:>10.1f}%")
    print(f"  FICO Score:           {fico:>10}")
    print(f"  Loan Purpose:         {purpose:>20}")
    print(f"  Term:                 {term:>20}")
    print(f"  Home Ownership:       {home_ownership:>20}")
    print("=" * 55)
    print("  MODEL PREDICTIONS")
    print("-" * 55)
    
    for name, prob in [('Logistic Regression', lr_prob), ('Decision Tree', dt_prob)]:
        risk_label = '🟢 LOW RISK' if prob < 0.25 else '🟡 MEDIUM RISK' if prob < 0.45 else '🔴 HIGH RISK'
        print(f"  {name:<25} {prob:.1%} default probability  {risk_label}")
    
    print("=" * 55)

print("✅ Scorer ready! Run the cells below to test borrowers.")

In [ ]:
# --- Borrower 1: Strong Profile ---
score_borrower(
    loan_amnt=10000,
    int_rate=0.07,
    annual_inc=95000,
    dti=8.5,
    fico=750,
    term=' 36 months',
    home_ownership='MORTGAGE',
    purpose='debt_consolidation',
    verification_status='Verified'
)

In [ ]:
# --- Borrower 2: Risky Profile ---
score_borrower(
    loan_amnt=35000,
    int_rate=0.25,
    annual_inc=38000,
    dti=38.0,
    fico=665,
    term=' 60 months',
    home_ownership='RENT',
    purpose='small_business',
    verification_status='Not Verified',
    delinq_2yrs=2,
    inq_last_6mths=4
)

In [ ]:
# ✏️ YOUR TURN — Try your own borrower!
# Change any values below and run the cell.

score_borrower(
    loan_amnt=15000,          # How much they want to borrow ($)
    int_rate=0.13,            # Interest rate (0.13 = 13%)
    annual_inc=65000,         # Annual income ($)
    dti=20.0,                 # Debt-to-income ratio (%)
    fico=700,                 # FICO credit score (660-850)
    term=' 36 months',        # ' 36 months' or ' 60 months'
    home_ownership='RENT',    # 'RENT', 'OWN', or 'MORTGAGE'
    purpose='credit_card',    # See list below
    delinq_2yrs=0,
    inq_last_6mths=2
)

# Valid purposes: 'debt_consolidation', 'credit_card', 'home_improvement',
#                 'other', 'major_purchase', 'small_business', 'medical',
#                 'car', 'moving', 'vacation', 'house'

---
## 💡 Step 11: Key Takeaways & Discussion

### What we built
Two machine learning models that predict loan default using only information available at the time of application — exactly what real fintech companies do.

### The precision vs. recall tradeoff — the most important business decision

There's no universally "correct" threshold. It depends on your business strategy:

| Strategy | Threshold | Effect |
|----------|-----------|--------|
| Conservative lender | Low (e.g. 25%) | Reject more loans, miss fewer defaulters, but turn away good customers |
| Aggressive lender | High (e.g. 60%) | Approve more loans, earn more interest, but absorb more defaults |
| Balanced | ~50% | Middle ground — what we used above |

This is ultimately a **business decision**, not a math decision. The model gives you the probability — you decide what to do with it.

### What would make this model better?
- More features (employment history, bank account data, spending patterns)
- Better models (Random Forest, Gradient Boosting — we kept it simple on purpose)
- More data and longer time horizon
- Separate models for different loan purposes or amounts

### What are the risks and ethical considerations?
- **Bias:** If the training data reflects historical discrimination, the model can perpetuate it
- **Explainability:** Regulations (like the Equal Credit Opportunity Act) require lenders to explain denied applications
- **Feedback loops:** If you reject people with low FICO scores, they never get the chance to build credit history
- **Proxy variables:** Zip code or loan purpose can be proxies for race or other protected characteristics

> 🏦 These are active debates in the fintech and banking industry. Machine learning is a powerful tool — but it comes with real responsibilities.

---
*Data: LendingClub loan data | Models: scikit-learn LogisticRegression, DecisionTreeClassifier*